<a href="https://colab.research.google.com/github/revianar/flyrank-ml-internship-starter/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

(https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

Do pages with weak measured search visibility still earn real engagement, and can that group be found automatically rather than by a hand-written threshold? Supports a weekly triage decision: whether a simple visibility flag is missing pages worth a discoverability push, for a strategist who can't audit every page by hand.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

FlyRank's pseudonymized warehouse, `fact_content_daily_performance`, March 2026 (outcome measured on April 2026, held out). 9,841,378 rows, 55 clients, 331,437 pages. Search Console coverage 36.7% of rows, Analytics 4.2% (3.7% both) — a three-valued flag, filtered `IS TRUE` rather than zero-filled. Engagement rate computable for 90,237/331,437 pages (27.2%); the rest excluded, not assumed zero. A 10-session floor leaves the modeled population: 21,207 pages, 32 clients, 6.4% of March pages. June 2026 exists in the warehouse but wasn't touched — reserved as a sealed check.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

Baseline: hand rule, session floor + Search Console availability ≤10% of days → 220 pages, ranked by March engagement. Model: K-Means on 4 standardized features (`engagement_rate`, `gsc_availability_rate`, `log_sessions`, `log_session_depth_sec`) — everything the baseline sees, plus depth. Validation: 5 client-grouped folds, two single-client; every label is out-of-fold. k selected by mean transfer ARI among k's where every held-out fold's smallest cluster holds ≥50 pages, fixed before inspecting results; silhouette shown for reference only (it rewards leakage — 0.654→0.712 when the cluster label is fed back as a feature). Rule selects k=3 (ARI 0.624); k=4 has nominally higher ARI (0.642) but is disqualified (smallest held-out cluster = 30 pages). "Hidden gem" = a cluster whose median engagement exceeds the population median (0.0000) and whose median visibility is ≤ the baseline's own 10% cutoff.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

No cluster, at k=3 or k=4, satisfies the hidden-gem definition — `GEM_CLUSTER = None` both times. But at k=4, out-of-fold agreement with the all-client clustering is high (ARI 0.847), and the resulting 689-page low-visibility cluster contains all 220 baseline-flagged pages and zero from any other cluster — independent, unsupervised confirmation the baseline threshold finds a real group. Permutation importance (held-out fold) explains the null: `log_sessions` (0.387) and `log_session_depth_sec` (0.138) drive cluster assignment; `gsc_availability_rate` (0.008) and `engagement_rate` (0.017) barely contribute.

## 5. Limitations

*What this work cannot claim.*

No model-selected set exists to formally beat the baseline — the finding is that the hand rule is validated, not that clustering outperforms it. The committed run uses k=4 by leftover override, not the rule's own k=3. The gradient split is post-hoc, same data that produced the null result. No control arm — persistence vs. a genuine visibility effect isn't disentangled. Top-N cuts are less reliable than the full set (regression to the mean, non-deterministic tie-breaking). 6.4% of March pages modeled; one feature month, one outcome month; no claim about search ranking or any algorithm.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

One supported action: keep using the existing 220-page flagged queue, not a model-generated replacement. Use the full set, not a top-20/50 cut — it's both better-performing and more stable across reruns. One reason code, `low_visibility_high_engagement` (a separate late-GSC-connection flag isn't warranted — only 0.5% of this set is explained by that). Before acting: confirm session count isn't at the floor's edge; check more than one prior month, since the worst top-20 misses were single standout months from concentrated clients.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
